<a href="https://colab.research.google.com/github/tangitapkullaniyor/CENG467_Midterm_290201060/blob/main/Question1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets transformers evaluate scikit-learn torch -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)
print(dataset["train"][0])

In [ ]:
from sklearn.model_selection import train_test_split

texts = list(dataset["train"]["text"])
labels = list(dataset["train"]["label"])

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

test_texts = list(dataset["test"]["text"])
test_labels = list(dataset["test"]["label"])

print("Train:", len(train_texts))
print("Validation:", len(val_texts))
print("Test:", len(test_texts))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)
X_test = vectorizer.transform(test_texts)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, train_labels)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

val_preds = model.predict(X_val)

val_acc = accuracy_score(val_labels, val_preds)
val_f1 = f1_score(val_labels, val_preds, average="macro")

print("Validation Accuracy:", val_acc)
print("Validation F1:", val_f1)

In [ ]:
test_preds = model.predict(X_test)

from sklearn.metrics import accuracy_score, f1_score

test_acc = accuracy_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, test_preds, average="macro")

print("Test Accuracy:", test_acc)
print("Test F1:", test_f1)

In [ ]:
val_preds = model.predict(X_val)

errors = []

for i in range(len(val_texts)):
    if val_labels[i] != val_preds[i]:
        errors.append(i)

print("Total errors:", len(errors))

In [ ]:
for i in errors[:5]:
    print("TEXT:", val_texts[i][:200])
    print("TRUE:", val_labels[i])
    print("PRED:", val_preds[i])
    print("-----")

In [ ]:
!pip install tensorflow -q

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models
import numpy as np
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
MAX_WORDS = 20000
MAX_LEN = 256

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

X_train_seq = tokenizer.texts_to_sequences(train_texts)
X_val_seq = tokenizer.texts_to_sequences(val_texts)
X_test_seq = tokenizer.texts_to_sequences(test_texts)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

y_train = np.array(train_labels)
y_val = np.array(val_labels)
y_test = np.array(test_labels)

print(X_train_pad.shape)
print(X_val_pad.shape)
print(X_test_pad.shape)

In [ ]:
model_bilstm = models.Sequential([
    layers.Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    layers.Bidirectional(layers.LSTM(64, dropout=0.3, recurrent_dropout=0.3)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

model_bilstm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_bilstm.summary()

In [ ]:
history = model_bilstm.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=3,
    batch_size=64
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

val_probs = model_bilstm.predict(X_val_pad)
val_preds_bilstm = (val_probs >= 0.5).astype(int).flatten()

test_probs = model_bilstm.predict(X_test_pad)
test_preds_bilstm = (test_probs >= 0.5).astype(int).flatten()

print("BiLSTM VAL ACC:", accuracy_score(y_val, val_preds_bilstm))
print("BiLSTM VAL F1:", f1_score(y_val, val_preds_bilstm, average="macro"))

print("BiLSTM TEST ACC:", accuracy_score(y_test, test_preds_bilstm))
print("BiLSTM TEST F1:", f1_score(y_test, test_preds_bilstm, average="macro"))

In [ ]:
errors_bilstm = []

for i in range(len(val_texts)):
    if y_val[i] != val_preds_bilstm[i]:
        errors_bilstm.append(i)

print("Total BiLSTM errors:", len(errors_bilstm))

In [ ]:
for i in errors_bilstm[:5]:
    print("TEXT:", val_texts[i][:200])
    print("TRUE:", y_val[i])
    print("PRED:", val_preds_bilstm[i])
    print("-----")

In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import numpy as np
from datasets import Dataset

In [ ]:
train_ds = Dataset.from_dict({
    "text": train_texts,
    "label": train_labels
})

val_ds = Dataset.from_dict({
    "text": val_texts,
    "label": val_labels
})

test_ds = Dataset.from_dict({
    "text": test_texts,
    "label": test_labels
})

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
model_bert = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    save_strategy="no",
    logging_steps=100,
    seed=42
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

In [ ]:
trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
val_results = trainer.evaluate()
print("BERT VAL:", val_results)

test_results = trainer.evaluate(test_ds)
print("BERT TEST:", test_results)

In [ ]:
pred_output = trainer.predict(val_ds)

logits = pred_output.predictions
labels = pred_output.label_ids

preds = np.argmax(logits, axis=1)

In [ ]:
errors_bert = []

for i in range(len(labels)):
    if labels[i] != preds[i]:
        errors_bert.append(i)

print("Total BERT errors:", len(errors_bert))

In [ ]:
for i in errors_bert[:5]:
    print("TEXT:", val_texts[i][:200])
    print("TRUE:", labels[i])
    print("PRED:", preds[i])
    print("-----")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer2 = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    stop_words=None
)

X_train2 = vectorizer2.fit_transform(train_texts)
X_val2 = vectorizer2.transform(val_texts)
X_test2 = vectorizer2.transform(test_texts)

from sklearn.linear_model import LogisticRegression

model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train2, train_labels)

val_preds2 = model2.predict(X_val2)
test_preds2 = model2.predict(X_test2)

from sklearn.metrics import accuracy_score, f1_score

print("VAL ACC:", accuracy_score(val_labels, val_preds2))
print("VAL F1:", f1_score(val_labels, val_preds2, average="macro"))

print("TEST ACC:", accuracy_score(test_labels, test_preds2))
print("TEST F1:", f1_score(test_labels, test_preds2, average="macro"))

In [ ]:
vectorizer_lower_false = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    stop_words=None,
    lowercase=False
)

X_train_lf = vectorizer_lower_false.fit_transform(train_texts)
X_val_lf = vectorizer_lower_false.transform(val_texts)
X_test_lf = vectorizer_lower_false.transform(test_texts)

model_lf = LogisticRegression(max_iter=1000)
model_lf.fit(X_train_lf, train_labels)

val_preds_lf = model_lf.predict(X_val_lf)
test_preds_lf = model_lf.predict(X_test_lf)

print("LOWERCASE FALSE VAL ACC:", accuracy_score(val_labels, val_preds_lf))
print("LOWERCASE FALSE VAL F1:", f1_score(val_labels, val_preds_lf, average="macro"))
print("LOWERCASE FALSE TEST ACC:", accuracy_score(test_labels, test_preds_lf))
print("LOWERCASE FALSE TEST F1:", f1_score(test_labels, test_preds_lf, average="macro"))

In [ ]:
MAX_LEN_128 = 128

X_train_pad_128 = pad_sequences(X_train_seq, maxlen=MAX_LEN_128, padding="post", truncating="post")
X_val_pad_128 = pad_sequences(X_val_seq, maxlen=MAX_LEN_128, padding="post", truncating="post")
X_test_pad_128 = pad_sequences(X_test_seq, maxlen=MAX_LEN_128, padding="post", truncating="post")

In [ ]:
model_bilstm_128 = models.Sequential([
    layers.Embedding(input_dim=MAX_WORDS, output_dim=128),
    layers.Bidirectional(layers.LSTM(64, dropout=0.3, recurrent_dropout=0.3)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

model_bilstm_128.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_128 = model_bilstm_128.fit(
    X_train_pad_128,
    y_train,
    validation_data=(X_val_pad_128, y_val),
    epochs=3,
    batch_size=64
)

In [ ]:
val_probs_128 = model_bilstm_128.predict(X_val_pad_128)
val_preds_128 = (val_probs_128 >= 0.5).astype(int).flatten()

test_probs_128 = model_bilstm_128.predict(X_test_pad_128)
test_preds_128 = (test_probs_128 >= 0.5).astype(int).flatten()

print("BiLSTM 128 VAL ACC:", accuracy_score(y_val, val_preds_128))
print("BiLSTM 128 VAL F1:", f1_score(y_val, val_preds_128, average="macro"))

print("BiLSTM 128 TEST ACC:", accuracy_score(y_test, test_preds_128))
print("BiLSTM 128 TEST F1:", f1_score(y_test, test_preds_128, average="macro"))